
# Raw BERT vs Internal Local Attention vs Internal Adaptive Attention

This notebook trains **3 raw models from scratch** on folder-based spam datasets:

1. **Raw BERT**
2. **Raw Internal Local Attention BERT**
3. **Raw Internal Adaptive Attention BERT**

## Folder format
```text
sms_root/
  spam/
  ham/

enron_root/
  spam/
  ham/
```

## Labels
- **spam = 0**
- **ham = 1**

## What it saves to Google Drive
- class counts
- train/val/test split CSVs
- training curves
- confusion matrices
- classification reports
- test metrics
- best model per dataset/model
- final summary CSV


## Cell 1 — Install packages

In [ ]:

!pip -q install transformers==4.39.3 scikit-learn pandas matplotlib seaborn tqdm pyyaml


## Cell 2 — Mount Google Drive

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


## Cell 3 — Imports

In [ ]:

import os
import re
import gc
import json
import math
import yaml
import random
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertConfig, BertModel, BertPreTrainedModel, AdamW, get_linear_schedule_with_warmup
from transformers.models.bert.modeling_bert import BertSelfAttention

warnings.filterwarnings("ignore")


## Cell 4 — Configuration

In [ ]:

CONFIG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "sms_file": "/content/drive/MyDrive/sms/SMSSpamCollection",
    "enron_root": "/content/drive/MyDrive/enron1",
    "output_root": "/content/drive/MyDrive/forensics_spam_raw_internal_outputs_V3",
    "train_size": 0.70,
    "val_size": 0.20,
    "test_size": 0.10,
    "batch_size": 16,
    "epochs": 6,
    "lr": 5e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "max_grad_norm": 1.0,
    "sms_max_len": 64,
    "enron_max_len": 128,
    "num_labels": 2,
    "local_window_size": 7,
    "vocab_size": 30522,
    "hidden_size": 768,
    "num_hidden_layers": 12,
    "num_attention_heads": 12,
    "intermediate_size": 3072,
    "hidden_dropout_prob": 0.1,
    "attention_probs_dropout_prob": 0.1,
    "allowed_extensions": [".txt", ".text", ".eml", ".msg"],
}
os.makedirs(CONFIG["output_root"], exist_ok=True)
with open(os.path.join(CONFIG["output_root"], "run_config.yaml"), "w") as f:
    yaml.safe_dump(CONFIG, f)
print("Device:", CONFIG["device"])
print("Output root:", CONFIG["output_root"])


## Cell 5 — Reproducibility

In [ ]:

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(CONFIG["seed"])


## Cell 6 — Data helpers

In [ ]:

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s@:/\.\-\$£€%!?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def list_text_files(folder: str, allowed_extensions: List[str]) -> List[str]:
    paths = []
    for root, _, files in os.walk(folder):
        for file in files:
            ext = os.path.splitext(file)[1].lower()
            if ext in allowed_extensions or ext == "":
                paths.append(os.path.join(root, file))
    return paths

def read_file_safely(path: str) -> str:
    encodings = ["utf-8", "latin-1", "cp1252", "utf-16"]
    for enc in encodings:
        try:
            with open(path, "r", encoding=enc, errors="ignore") as f:
                return f.read()
        except Exception:
            continue
    return ""

def load_folder_dataset(dataset_root: str, dataset_name: str) -> pd.DataFrame:
    spam_dir = os.path.join(dataset_root, "spam")
    ham_dir = os.path.join(dataset_root, "ham")
    assert os.path.isdir(spam_dir), f"Missing folder: {spam_dir}"
    assert os.path.isdir(ham_dir), f"Missing folder: {ham_dir}"
    spam_files = list_text_files(spam_dir, CONFIG["allowed_extensions"])
    ham_files = list_text_files(ham_dir, CONFIG["allowed_extensions"])
    records = []
    for path in tqdm(spam_files, desc=f"Loading {dataset_name} spam"):
        txt = clean_text(read_file_safely(path))
        if txt:
            records.append({"text": txt, "label": 0, "label_name": "spam", "path": path})
    for path in tqdm(ham_files, desc=f"Loading {dataset_name} ham"):
        txt = clean_text(read_file_safely(path))
        if txt:
            records.append({"text": txt, "label": 1, "label_name": "ham", "path": path})
    df = pd.DataFrame(records).drop_duplicates(subset=["text"]).reset_index(drop=True)
    print(f"\n[{dataset_name}] Total samples: {len(df)}")
    print(f"[{dataset_name}] spam (label=0): {(df['label'] == 0).sum()}")
    print(f"[{dataset_name}] ham  (label=1): {(df['label'] == 1).sum()}")
    return df

def load_sms_collection_file(file_path):
    texts = []
    labels = []

    with open(file_path, "r", encoding="latin-1") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) != 2:
                continue

            label_text, message = parts
            message = clean_text(message)

            if label_text.lower() == "spam":
                label = 0
            elif label_text.lower() == "ham":
                label = 1
            else:
                continue

            texts.append(message)
            labels.append(label)

    df = pd.DataFrame({
        "text": texts,
        "label": labels
    })

    df["label_name"] = df["label"].map({0: "spam", 1: "ham"})

    print(f"\n[sms] Total samples: {len(df)}")
    print(f"[sms] spam (label=0): {(df['label'] == 0).sum()}")
    print(f"[sms] ham  (label=1): {(df['label'] == 1).sum()}")

    return df

def stratified_split(df: pd.DataFrame, seed: int = 42):
    train_df, temp_df = train_test_split(df, test_size=(1 - CONFIG["train_size"]), stratify=df["label"], random_state=seed)
    val_ratio_from_temp = CONFIG["val_size"] / (CONFIG["val_size"] + CONFIG["test_size"])
    val_df, test_df = train_test_split(temp_df, test_size=(1 - val_ratio_from_temp), stratify=temp_df["label"], random_state=seed)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

def save_split_csvs(train_df, val_df, test_df, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    train_df.to_csv(os.path.join(out_dir, "train.csv"), index=False)
    val_df.to_csv(os.path.join(out_dir, "val.csv"), index=False)
    test_df.to_csv(os.path.join(out_dir, "test.csv"), index=False)

def save_class_distribution_plot(df: pd.DataFrame, dataset_name: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    counts = df["label_name"].value_counts()
    plt.figure(figsize=(6, 4))
    sns.barplot(x=counts.index, y=counts.values)
    plt.title(f"{dataset_name} Class Distribution")
    plt.ylabel("Count")
    plt.xlabel("Class")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{dataset_name}_class_distribution.png"), dpi=200)
    plt.show()
    plt.close()


## Cell 7 — Load SMS and Enron, print class counts, split, save CSVs

In [ ]:

sms_df = load_sms_collection_file(CONFIG["sms_file"])
enron_df = load_folder_dataset(CONFIG["enron_root"], "enron")

save_class_distribution_plot(sms_df, "sms", os.path.join(CONFIG["output_root"], "sms"))
save_class_distribution_plot(enron_df, "enron", os.path.join(CONFIG["output_root"], "enron"))

sms_train, sms_val, sms_test = stratified_split(sms_df, CONFIG["seed"])
enron_train, enron_val, enron_test = stratified_split(enron_df, CONFIG["seed"])

save_split_csvs(sms_train, sms_val, sms_test, os.path.join(CONFIG["output_root"], "sms", "splits"))
save_split_csvs(enron_train, enron_val, enron_test, os.path.join(CONFIG["output_root"], "enron", "splits"))

print("SMS split sizes:", len(sms_train), len(sms_val), len(sms_test))
print("Enron split sizes:", len(enron_train), len(enron_val), len(enron_test))


## Cell 8 — Tokenizer, dataset class, dataloaders

In [ ]:

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class TextDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def create_dataloaders(train_df, val_df, test_df, tokenizer, max_len, batch_size):
    train_ds = TextDataset(train_df, tokenizer, max_len)
    val_ds = TextDataset(val_df, tokenizer, max_len)
    test_ds = TextDataset(test_df, tokenizer, max_len)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    return train_loader, val_loader, test_loader


## Cell 9 — Raw BERT, internal Local Attention, internal Adaptive Attention

In [ ]:

class RawBaselineBERT(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.bert = BertModel(config)
        classifier_dropout = config.classifier_dropout if getattr(config, "classifier_dropout", None) is not None else config.hidden_dropout_prob
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(pooled))
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

class LocalBertSelfAttention(BertSelfAttention):
    def __init__(self, config, position_embedding_type=None, window_size=7):
        super().__init__(config, position_embedding_type=position_embedding_type)
        self.window_size = window_size

    def _distance_mask(self, seq_len, device, dtype):
        idx = torch.arange(seq_len, device=device)
        dist = (idx[None, :] - idx[:, None]).abs()
        allowed = dist <= self.window_size
        mask = torch.zeros(seq_len, seq_len, device=device, dtype=dtype)
        mask = mask.masked_fill(~allowed, torch.finfo(dtype).min)
        return mask.view(1, 1, seq_len, seq_len)

    def forward(self, hidden_states, attention_mask=None, head_mask=None, encoder_hidden_states=None, encoder_attention_mask=None, past_key_value=None, output_attentions=False):
        mixed_query_layer = self.query(hidden_states)
        is_cross_attention = encoder_hidden_states is not None
        current_states = encoder_hidden_states if is_cross_attention else hidden_states
        applied_attention_mask = encoder_attention_mask if is_cross_attention else attention_mask
        key_layer = self.transpose_for_scores(self.key(current_states))
        value_layer = self.transpose_for_scores(self.value(current_states))
        query_layer = self.transpose_for_scores(mixed_query_layer)
        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        seq_len = attention_scores.size(-1)
        attention_scores = attention_scores + self._distance_mask(seq_len, attention_scores.device, attention_scores.dtype)
        if applied_attention_mask is not None:
            attention_scores = attention_scores + applied_attention_mask
        attention_probs = nn.functional.softmax(attention_scores, dim=-1)
        attention_probs = self.dropout(attention_probs)
        if head_mask is not None:
            attention_probs = attention_probs * head_mask
        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        context_layer = context_layer.view(context_layer.size()[:-2] + (self.all_head_size,))
        outputs = (context_layer, attention_probs) if output_attentions else (context_layer,)
        if self.is_decoder:
            outputs = outputs + ((key_layer, value_layer),)
        return outputs

class AdaptiveBertSelfAttention(BertSelfAttention):
    def __init__(self, config, position_embedding_type=None, window_size=7):
        super().__init__(config, position_embedding_type=position_embedding_type)
        self.window_size = window_size
        self.alpha_gate = nn.Linear(config.hidden_size, 1)

    def _distance_mask(self, seq_len, device, dtype):
        idx = torch.arange(seq_len, device=device)
        dist = (idx[None, :] - idx[:, None]).abs()
        allowed = dist <= self.window_size
        mask = torch.zeros(seq_len, seq_len, device=device, dtype=dtype)
        mask = mask.masked_fill(~allowed, torch.finfo(dtype).min)
        return mask.view(1, 1, seq_len, seq_len)

    def forward(self, hidden_states, attention_mask=None, head_mask=None, encoder_hidden_states=None, encoder_attention_mask=None, past_key_value=None, output_attentions=False):
        mixed_query_layer = self.query(hidden_states)
        is_cross_attention = encoder_hidden_states is not None
        current_states = encoder_hidden_states if is_cross_attention else hidden_states
        applied_attention_mask = encoder_attention_mask if is_cross_attention else attention_mask
        key_layer = self.transpose_for_scores(self.key(current_states))
        value_layer = self.transpose_for_scores(self.value(current_states))
        query_layer = self.transpose_for_scores(mixed_query_layer)
        base_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        base_scores = base_scores / math.sqrt(self.attention_head_size)

        global_scores = base_scores
        if applied_attention_mask is not None:
            global_scores = global_scores + applied_attention_mask
        global_probs = nn.functional.softmax(global_scores, dim=-1)

        seq_len = base_scores.size(-1)
        local_scores = base_scores + self._distance_mask(seq_len, base_scores.device, base_scores.dtype)
        if applied_attention_mask is not None:
            local_scores = local_scores + applied_attention_mask
        local_probs = nn.functional.softmax(local_scores, dim=-1)

        cls_state = hidden_states[:, 0, :]
        alpha = torch.sigmoid(self.alpha_gate(cls_state)).view(-1, 1, 1, 1)
        attention_probs = alpha * local_probs + (1 - alpha) * global_probs
        attention_probs = self.dropout(attention_probs)
        if head_mask is not None:
            attention_probs = attention_probs * head_mask
        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        context_layer = context_layer.view(context_layer.size()[:-2] + (self.all_head_size,))
        outputs = (context_layer, attention_probs) if output_attentions else (context_layer,)
        if self.is_decoder:
            outputs = outputs + ((key_layer, value_layer),)
        return outputs

class InternalLocalAttentionBERT(BertPreTrainedModel):
    def __init__(self, config, window_size=7):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.bert = BertModel(config)
        classifier_dropout = config.classifier_dropout if getattr(config, "classifier_dropout", None) is not None else config.hidden_dropout_prob
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        for i in range(len(self.bert.encoder.layer)):
            self.bert.encoder.layer[i].attention.self = LocalBertSelfAttention(config, position_embedding_type=getattr(config, "position_embedding_type", None), window_size=window_size)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(pooled))
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

class InternalAdaptiveAttentionBERT(BertPreTrainedModel):
    def __init__(self, config, window_size=7):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.bert = BertModel(config)
        classifier_dropout = config.classifier_dropout if getattr(config, "classifier_dropout", None) is not None else config.hidden_dropout_prob
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        for i in range(len(self.bert.encoder.layer)):
            self.bert.encoder.layer[i].attention.self = AdaptiveBertSelfAttention(config, position_embedding_type=getattr(config, "position_embedding_type", None), window_size=window_size)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(pooled))
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

def base_raw_config(num_labels):
    return BertConfig(
        vocab_size=CONFIG["vocab_size"],
        hidden_size=CONFIG["hidden_size"],
        num_hidden_layers=CONFIG["num_hidden_layers"],
        num_attention_heads=CONFIG["num_attention_heads"],
        intermediate_size=CONFIG["intermediate_size"],
        hidden_dropout_prob=CONFIG["hidden_dropout_prob"],
        attention_probs_dropout_prob=CONFIG["attention_probs_dropout_prob"],
        num_labels=num_labels,
    )

def build_raw_bert(num_labels):
    return RawBaselineBERT(base_raw_config(num_labels))

def build_raw_local_model(num_labels, window_size=7):
    return InternalLocalAttentionBERT(base_raw_config(num_labels), window_size=window_size)

def build_raw_adaptive_model(num_labels, window_size=7):
    return InternalAdaptiveAttentionBERT(base_raw_config(num_labels), window_size=window_size)


In [ ]:
print(base_raw_config(2))

## Cell 10 — Metrics and plots

In [ ]:

def compute_metrics(y_true, y_pred) -> Dict[str, float]:
    Accuracy = accuracy_score(y_true, y_pred)
    Precision, Recall, F1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    Weighted_Precision, Weighted_Recall, Weighted_F1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "accuracy": Accuracy,
        "precision": Precision,
        "recall": Recall,
        "f1": F1,
        "weighted_precision": Weighted_Precision,
        "weighted_recall": Weighted_Recall,
        "weighted_f1": Weighted_F1,
    }

def plot_training_curves(history, title_prefix, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for y1, y2, name, ylabel in [
        ("train_loss", "val_loss", "loss_curve", "Loss"),
        ("train_accuracy", "val_accuracy", "accuracy_curve", "Accuracy"),
        ("train_f1", "val_f1", "f1_curve", "F1"),
    ]:
        plt.figure(figsize=(7, 5))
        plt.plot(history[y1], label=y1)
        plt.plot(history[y2], label=y2)
        plt.title(f"{title_prefix} - {name}")
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{name}.png"), dpi=200)
        plt.show()
        plt.close()

def plot_confusion_matrix(y_true, y_pred, title, out_dir):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["spam(0)", "ham(1)"], yticklabels=["spam(0)", "ham(1)"])
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=200)
    plt.show()
    plt.close()

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


## Cell 11 — Training and evaluation

In [ ]:

def get_optimizer_and_scheduler(model, train_loader_len):
    optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    total_steps = train_loader_len * CONFIG["epochs"]
    warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    return optimizer, scheduler

def run_epoch(model, loader, optimizer, scheduler, device, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for batch in tqdm(loader, leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs["loss"]
            logits = outputs["logits"]
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
                optimizer.step()
                scheduler.step()
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())
    metrics = compute_metrics(all_labels, all_preds)
    metrics["loss"] = total_loss / len(loader)
    return metrics, all_labels, all_preds

def save_best_model(model, tokenizer, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))
    if hasattr(model, "config"):
        model.config.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

def train_model(model, model_name, dataset_name, train_loader, val_loader, test_loader, tokenizer):
    device = CONFIG["device"]
    model = model.to(device)
    run_dir = os.path.join(CONFIG["output_root"], dataset_name, model_name)
    os.makedirs(run_dir, exist_ok=True)
    optimizer, scheduler = get_optimizer_and_scheduler(model, len(train_loader))
    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": [], "train_f1": [], "val_f1": []}
    best_val_f1 = -1.0
    best_epoch = -1
    best_model_dir = os.path.join(run_dir, "best_model")

    for epoch in range(CONFIG["epochs"]):
        print(f"\n===== {dataset_name.upper()} | {model_name} | Epoch {epoch+1}/{CONFIG['epochs']} =====")
        train_metrics, _, _ = run_epoch(model, train_loader, optimizer, scheduler, device, train=True)
        val_metrics, _, _ = run_epoch(model, val_loader, optimizer, scheduler, device, train=False)

        history["train_loss"].append(train_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["train_accuracy"].append(train_metrics["accuracy"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["train_f1"].append(train_metrics["f1"])
        history["val_f1"].append(val_metrics["f1"])

        print("Train:", {k: round(v, 4) for k, v in train_metrics.items()})
        print("Val  :", {k: round(v, 4) for k, v in val_metrics.items()})

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_epoch = epoch + 1
            save_best_model(model, tokenizer, best_model_dir)

    plot_training_curves(history, f"{dataset_name}-{model_name}", run_dir)
    save_json(history, os.path.join(run_dir, "training_history.json"))

    test_metrics, y_true, y_pred = run_epoch(model, test_loader, None, None, device, train=False)
    print("\nTest metrics:", {k: round(v, 4) for k, v in test_metrics.items()})
    plot_confusion_matrix(y_true, y_pred, f"{dataset_name}-{model_name} Test Confusion Matrix", run_dir)
    save_json(test_metrics, os.path.join(run_dir, "test_metrics.json"))

    with open(os.path.join(run_dir, "classification_report.txt"), "w") as f:
        f.write(classification_report(y_true, y_pred, target_names=["spam(0)", "ham(1)"], digits=4, zero_division=0))

    return {"dataset": dataset_name, "model": model_name, "best_val_f1": best_val_f1, "best_epoch": best_epoch, **test_metrics}


## Cell 12 — Run the 3 raw models on one dataset

In [ ]:

def run_experiments_for_dataset(dataset_name, train_df, val_df, test_df, max_len, tokenizer):
    train_loader, val_loader, test_loader = create_dataloaders(train_df, val_df, test_df, tokenizer, max_len, CONFIG["batch_size"])
    summaries = []

    model = build_raw_bert(CONFIG["num_labels"])
    summaries.append(train_model(model, "raw_bert", dataset_name, train_loader, val_loader, test_loader, tokenizer))
    del model
    gc.collect()
    torch.cuda.empty_cache()

    model = build_raw_local_model(CONFIG["num_labels"], CONFIG["local_window_size"])
    summaries.append(train_model(model, "raw_internal_local_attention", dataset_name, train_loader, val_loader, test_loader, tokenizer))
    del model
    gc.collect()
    torch.cuda.empty_cache()

    model = build_raw_adaptive_model(CONFIG["num_labels"], CONFIG["local_window_size"])
    summaries.append(train_model(model, "raw_internal_adaptive_attention", dataset_name, train_loader, val_loader, test_loader, tokenizer))
    del model
    gc.collect()
    torch.cuda.empty_cache()

    return pd.DataFrame(summaries)


## Cell 13 — Run all experiments

In [ ]:

sms_results = run_experiments_for_dataset("sms", sms_train, sms_val, sms_test, CONFIG["sms_max_len"], tokenizer)
enron_results = run_experiments_for_dataset("enron", enron_train, enron_val, enron_test, CONFIG["enron_max_len"], tokenizer)

all_results = pd.concat([sms_results, enron_results], ignore_index=True)
all_results.to_csv(os.path.join(CONFIG["output_root"], "all_results_summary.csv"), index=False)

print(all_results)
print("\nAll outputs saved to:", CONFIG["output_root"])


## Cell 14 — Reload summary later

In [ ]:

summary_path = os.path.join(CONFIG["output_root"], "all_results_summary.csv")
if os.path.exists(summary_path):
    display(pd.read_csv(summary_path))
else:
    print("Summary file not found yet. Run Cell 13 first.")
